# 手撕 Flash Attention Pytorch实现

![](./flash.png)

## 1. 设定分块矩阵

In [1]:
import torch
from einops import rearrange
torch.manual_seed(42)

NEG_INF = -1e10  # -infinity
EPSILON = 1e-10

Q_LEN = 6
K_LEN = 6
Q_BLOCK_SIZE = 3 # 
KV_BLOCK_SIZE = 3
Tr = Q_LEN // Q_BLOCK_SIZE
Tc = K_LEN // KV_BLOCK_SIZE

Q = torch.randn(1, 1, Q_LEN, 4, requires_grad=True).to(device='cpu')
K = torch.randn(1, 1, K_LEN, 4, requires_grad=True).to(device='cpu')
V = torch.randn(1, 1, K_LEN, 4, requires_grad=True).to(device='cpu')

In [2]:
O = torch.softmax ( Q @ K.transpose(2,3), dim = -1) @ V
print(O)

tensor([[[[-0.3839,  0.3844,  0.0229, -0.3405],
          [-0.0877,  0.1931,  2.7363,  0.8162],
          [-0.1728,  0.3630,  1.7052,  0.7037],
          [-0.1631,  0.2604,  3.6420,  1.1873],
          [-0.2228, -0.0282, -0.5896, -0.3328],
          [-0.2500,  0.3592,  1.4500,  0.7168]]]],
       grad_fn=<UnsafeViewBackward0>)


## 2. 计算Flash Attention

In [3]:
O = torch.zeros_like(Q, requires_grad=True)
l = torch.zeros(Q.shape[:-1])[..., None]
m = torch.ones(Q.shape[:-1])[..., None] * NEG_INF

Q_BLOCKS = torch.split(Q, Q_BLOCK_SIZE, dim=2)
K_BLOCKS = torch.split(K, KV_BLOCK_SIZE, dim=2)
V_BLOCKS = torch.split(V, KV_BLOCK_SIZE, dim=2)
O_BLOCKS = list(torch.split(O, Q_BLOCK_SIZE, dim=2))
l_BLOCKS = list(torch.split(l, Q_BLOCK_SIZE, dim=2))
m_BLOCKS = list(torch.split(m, Q_BLOCK_SIZE, dim=2))

# 先 KV 后 Q 
for j in range(Tc):
    Kj = K_BLOCKS[j]
    Vj = V_BLOCKS[j]
    for i in range(Tr):
        Qi = Q_BLOCKS[i]
        Oi = O_BLOCKS[i]
        li = l_BLOCKS[i]
        mi = m_BLOCKS[i]

        S_ij = Qi @ Kj.transpose(2,3)
        m_block_ij, _ = torch.max(S_ij, dim=-1, keepdims=True)
        P_ij = torch.exp(S_ij - m_block_ij)
        l_block_ij = torch.sum(P_ij, dim=-1, keepdims=True) + EPSILON
        mi_new = torch.maximum(m_block_ij, mi)
        P_ij_Vj = P_ij @ Vj
        
        li_new = torch.exp(mi - mi_new) * li  \
               + torch.exp(m_block_ij - mi_new) * l_block_ij 

        O_BLOCKS[i] = (li / li_new) * torch.exp(mi - mi_new) * Oi \
                    +(torch.exp(m_block_ij - mi_new) / li_new) * P_ij_Vj
        print(f'-----------Attn : Q{i}xK{j}---------')
#         print(O_BLOCKS[i].shape)
        print(O_BLOCKS[0])
        print(O_BLOCKS[1])
        print('\n')
        
        l_BLOCKS[i] = li_new
        m_BLOCKS[i] = mi_new

O = torch.cat(O_BLOCKS, dim=2)
l = torch.cat(l_BLOCKS, dim=2)
m = torch.cat(m_BLOCKS, dim=2)

print(O)

-----------Attn : Q0xK0---------
tensor([[[[-0.3828,  0.3858,  0.0073, -0.3497],
          [ 0.1703,  0.0784, -0.2246, -0.3114],
          [ 0.2711,  0.4141,  0.6492, -0.1008]]]], grad_fn=<AddBackward0>)
tensor([[[[0., 0., 0., 0.],
          [0., 0., 0., 0.],
          [0., 0., 0., 0.]]]], grad_fn=<SplitBackward0>)


-----------Attn : Q1xK0---------
tensor([[[[-0.3828,  0.3858,  0.0073, -0.3497],
          [ 0.1703,  0.0784, -0.2246, -0.3114],
          [ 0.2711,  0.4141,  0.6492, -0.1008]]]], grad_fn=<AddBackward0>)
tensor([[[[ 0.5309,  0.5167,  1.1180,  0.0456],
          [-0.1518, -0.0681, -0.8508, -0.5029],
          [ 0.3779,  0.3903,  0.6877, -0.0749]]]], grad_fn=<AddBackward0>)


-----------Attn : Q0xK1---------
tensor([[[[-0.3839,  0.3844,  0.0229, -0.3405],
          [-0.0877,  0.1931,  2.7363,  0.8162],
          [-0.1728,  0.3630,  1.7052,  0.7037]]]], grad_fn=<AddBackward0>)
tensor([[[[ 0.5309,  0.5167,  1.1180,  0.0456],
          [-0.1518, -0.0681, -0.8508, -0.5029],
    

# Flash Attention2

Flash Attntion2 相较1最终要的特点就是改变内外循环，从而减少O的交换

![](./flash2.png)

Flash Attention 每个步骤都要做scaled处理， Flash Attention2 只在最后做scaled

$$
\begin{align} O^{(2)}&=diag(l^{(1)}/l^{(2)})^{-1}O^{(1)}+diag(l^{(2)})^{-1}e^{S^{(2)}-m^{(2)}}V^{(2)} \end{align}
$$

![](./flash_scaled.png)_

$$
\begin{align} \widetilde{O}^{(2)} &= diag(l^{(1)})^{-1}O^{(1)}+e^{S^{(2)}-m^{(2)}}V^{(2)} \\ O^{(2)} &= diag(l^{(2)})^{-1}\widetilde{O}^{(2)} \\  O^{(N)} &= diag(l^{(N)})^{-1}\widetilde{O}^{(N)} \end{align}
$$

In [4]:
O = torch.zeros_like(Q, requires_grad=True)
l = torch.zeros(Q.shape[:-1])[..., None]
# l_cache = torch.zeros(Q.shape[:-1])[..., None] + 1.0
m = torch.ones(Q.shape[:-1])[..., None] * NEG_INF

O_BLOCKS = list(torch.split(O, Q_BLOCK_SIZE, dim=2))
l_BLOCKS = list(torch.split(l, Q_BLOCK_SIZE, dim=2))
# l_cache_BLOCKS = list(torch.split(l_cache, Q_BLOCK_SIZE, dim=2))
m_BLOCKS = list(torch.split(m, Q_BLOCK_SIZE, dim=2))

# start with Q
for i in range(Tr):
    Qi = Q_BLOCKS[i]
    Oi = O_BLOCKS[i]
    li = l_BLOCKS[i]
    mi = m_BLOCKS[i]
    # li_cache = l_cache_BLOCKS[i]
    
    for j in range(Tc):
        #if j>i: 
        #    continue    # ignore masked      
        Kj = K_BLOCKS[j]
        Vj = V_BLOCKS[j]

        S_ij = Qi @ Kj.transpose(2,3)
        m_block_ij, _ = torch.max(S_ij, dim=-1, keepdims=True)
        mi_new = torch.maximum(m_block_ij, mi)
        P_ij_hat = torch.exp(S_ij - mi_new)
        l_block_ij = torch.sum(P_ij_hat, dim=-1, keepdims=True) + EPSILON
        
        li_new = torch.exp(mi - mi_new) * li  + l_block_ij 
        Oi = torch.exp(mi - mi_new) * Oi + P_ij_hat @ Vj   
        
        li = li_new
        mi = mi_new
        print(f'-----------O{i} = attn( Q{i}, KV[{j}])---------')
        print(Oi)
        
    O_BLOCKS[i] = Oi / li_new # 最后做Scaled
    l_BLOCKS[i] = li_new
    m_BLOCKS[i] = mi_new
    
O = torch.cat(O_BLOCKS, dim=2)
l = torch.cat(l_BLOCKS, dim=2)
m = torch.cat(m_BLOCKS, dim=2)

print(O)

-----------O0 = attn( Q0, KV[0])---------
tensor([[[[-0.5712,  0.5756,  0.0109, -0.5217],
          [ 0.3267,  0.1504, -0.4310, -0.5974],
          [ 0.3851,  0.5882,  0.9223, -0.1432]]]], grad_fn=<AddBackward0>)
-----------O0 = attn( Q0, KV[1])---------
tensor([[[[-0.5769,  0.5776,  0.0344, -0.5117],
          [-0.1262,  0.2778,  3.9363,  1.1742],
          [-0.3885,  0.8162,  3.8338,  1.5820]]]], grad_fn=<AddBackward0>)
-----------O1 = attn( Q1, KV[0])---------
tensor([[[[ 0.5361,  0.5217,  1.1288,  0.0461],
          [-0.2164, -0.0970, -1.2126, -0.7168],
          [ 0.4848,  0.5008,  0.8824, -0.0961]]]], grad_fn=<AddBackward0>)
-----------O1 = attn( Q1, KV[1])---------
tensor([[[[-0.1802,  0.2877,  4.0247,  1.3121],
          [-0.3674, -0.0465, -0.9725, -0.5489],
          [-0.6276,  0.9017,  3.6402,  1.7995]]]], grad_fn=<AddBackward0>)
tensor([[[[-0.3839,  0.3844,  0.0229, -0.3405],
          [-0.0877,  0.1931,  2.7363,  0.8162],
          [-0.1728,  0.3630,  1.7052,  0.7037],
    